In [33]:
using ArgParse
using Dates
using JLD2
using Printf

using StatsBase
using Graphs
using SimpleWeightedGraphs
using GraphsMatching
using GLMakie

include("../src/circuits.jl")
include("../src/local.jl")
include("../src/main.jl")
include("../src/mwpm.jl")


basic_correct (generic function with 1 method)

In [34]:
function adv_MWPM_sample_hotstart(ρ0::Array{Bool,2}, L::Int, T::Int, p::Float64, peff::Float64, q::Float64)
    ρ = deepcopy(ρ0)
    Ms = zeros(Bool, 2T+2, L, L)
    horizontal_checks = zeros(Bool, 2T+2, L, L)
    vertical_checks = zeros(Bool, 2T+2, L, L)

    for t in 1:T
        ρ = noiselayer(ρ, p)
        Ms[2t+1, :, :] = ρ
        ρ, original_checks, new_checks = correct(ρ, peff, q)
        horizontal_checks[2t+1, :, :] = original_checks[1]
        vertical_checks[2t+1, :, :] = original_checks[2]
        horizontal_checks[2t+2, :, :] = new_checks[1]
        vertical_checks[2t+2, :, :] = new_checks[2]
        Ms[2t+2, :, :] = ρ
    end

    return Ms, horizontal_checks, vertical_checks
end

function correct(ρ::AbstractMatrix, peff::Float64, q::Float64)
    checks = measure(ρ, q)
    original_checks = deepcopy(checks)
    
    horizontal_checks, vertical_checks = heal(checks, peff, q)
    domain = track_domains((horizontal_checks, vertical_checks))
    if magnetization(domain) > 0.5
        domain = .!domain
    end
    
    return ρ .⊻ domain, original_checks, (horizontal_checks, vertical_checks)
end

correct (generic function with 1 method)

In [62]:
L = 32
ℓ = 8
p = 0.05
q = 0.1
peff = 0.5
T = 2L


ρ = zeros(Bool, L, L)
ρ[L÷2-ℓ÷2:L÷2+ℓ÷2,L÷2-ℓ÷2:L÷2+ℓ÷2] .= true


M1s, horizontal_checks, vertical_checks = adv_MWPM_sample_hotstart(ρ, L, T, p, peff, q)
M1s[1,:,:] = ρ
M1s[2,:,:] = ρ;

In [61]:
GLMakie.activate!()
GLMakie.closeall()

# =========================
# STYLE SETTINGS
# =========================
HEAT_COLORMAP  = :viridis        # e.g. :viridis, :magma, :plasma, :grays
HEAT_RANGE     = nothing         # nothing = auto from data, or set (min,max)

EDGE_COLOR     = :white          # e.g. :white, :black, :red
EDGE_WIDTH     = 3.0             # thickness of overlay edges
# =========================


# -------------------------
# 1) Time-stretch data
# -------------------------
Nt_old, Lx, Ly = size(M1s)
Nt = 2Nt_old
K  = Nt ÷ 4

M  = similar(M1s, Nt, Lx, Ly)
H  = similar(horizontal_checks, Nt, Lx, Ly)
V  = similar(vertical_checks,   Nt, Lx, Ly)

for k in 1:K
    t0 = 4k - 3
    M[t0:t0+2, :, :] .= reshape(M1s[2k-1, :, :], 1, Lx, Ly)
    M[t0+3,   :, :]  .= M1s[2k, :, :]
end

fill!(H, false)
fill!(V, false)
for k in 1:K
    t0 = 4k - 3
    H[t0+1, :, :]        .= horizontal_checks[2k-1, :, :]
    V[t0+1, :, :]        .= vertical_checks[2k-1, :, :]
    H[t0+2:t0+3, :, :]   .= reshape(horizontal_checks[2k, :, :], 1, Lx, Ly)
    V[t0+2:t0+3, :, :]   .= reshape(vertical_checks[2k, :, :],   1, Lx, Ly)
end


# -------------------------
# 2) Checks -> line segments
# -------------------------
function edge_segments(hc::AbstractMatrix, vc::AbstractMatrix)
    Lx, Ly = size(hc)
    pts = Point2f[]

    for x in 1:(Lx-1), y in 1:Ly
        if hc[x,y]
            x0 = x + 0.5f0
            push!(pts, Point2f(x0, y-0.5f0), Point2f(x0, y+0.5f0))
        end
    end

    for x in 1:Lx, y in 1:(Ly-1)
        if vc[x,y]
            y0 = y + 0.5f0
            push!(pts, Point2f(x-0.5f0, y0), Point2f(x+0.5f0, y0))
        end
    end

    return pts
end


# -------------------------
# 3) UI
# -------------------------
fig = Figure(size=(900, 900))
ax  = Axis(fig[1, 1:4])

btn_prev = Button(fig[2, 1], label="◀")
sl       = Slider(fig[2, 2], range=1:Nt, startvalue=1)
btn_next = Button(fig[2, 3], label="▶")
lbl      = Label(fig[2, 4], "t = 1")
colsize!(fig.layout, 2, Relative(1))

# Heatmap observable
heat = Observable(M[1, :, :])

cr = HEAT_RANGE === nothing ? extrema(M) : HEAT_RANGE

GLMakie.heatmap!(
    ax,
    heat;
    colormap  = HEAT_COLORMAP,
    colorrange = cr
)

# Edge overlay
edges = Observable(edge_segments(H[1, :, :], V[1, :, :]))

linesegments!(
    ax,
    edges;
    color     = EDGE_COLOR,
    linewidth = EDGE_WIDTH
)

# ---- time state ----
t = Observable(1)
syncing = Ref(false)

on(sl.value) do x
    syncing[] && return
    syncing[] = true
    t[] = clamp(round(Int, x), 1, Nt)
    syncing[] = false
end

on(t) do tt
    heat[]  = M[tt, :, :]
    edges[] = edge_segments(H[tt, :, :], V[tt, :, :])
    lbl.text[] = "t = $tt"

    syncing[] = true
    Makie.set_close_to!(sl, tt)
    syncing[] = false
end

step!(δ) = (t[] = clamp(t[] + δ, 1, Nt))

on(btn_prev.clicks) do _; step!(-1); end
on(btn_next.clicks) do _; step!(+1); end

on(events(fig).keyboardbutton) do ev
    if ev.action == Keyboard.press || ev.action == Keyboard.repeat
        ev.key == Keyboard.left  && step!(-1)
        ev.key == Keyboard.right && step!(+1)
    end
end

display(fig)

GLMakie.Screen(...)